<a href="https://colab.research.google.com/github/ohita-dev/ohita-cookbooks/blob/main/Ohita_LangChain_Quickstart.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Welcome to the Ohita Quickstart! In 60 seconds, you'll see how easy it is to give an LLM access to external APIs using one Ohita key.**

In [ ]:
# Install LangChain, the OpenAI integration, and requests
!pip install -qU langchain langchain-openai requests

In [ ]:
import getpass
import os

print("🔑 Enter your OpenAI API Key (starts with sk-...):")
os.environ["OPENAI_API_KEY"] = getpass.getpass()

print("\n🔑 Enter your Ohita API Key (Get one instantly at https://ohita.tech):")
os.environ["OHITA_API_KEY"] = getpass.getpass()

print("\n✅ Keys securely loaded into the temporary environment!")

In [ ]:
import requests
from langchain_core.tools import tool

# The @tool decorator automatically registers this function as a tool the LLM can use
@tool
def get_hacker_news_top_stories() -> str:
    """Fetches the top stories from Hacker News. Use this when the user asks about Hacker News."""
    url = "https://api.ohita.tech/v1/hackernews/top"
    headers = {"Authorization": f"Bearer {os.environ['OHITA_API_KEY']}"}

    try:
        response = requests.get(url, headers=headers)
        response.raise_for_status()

        # We return the data as a string so the LLM can easily read and parse it
        return str(response.json())
    except Exception as e:
        return f"Error fetching data from Ohita: {e}"

# List of tools we will give to the agent
tools = [get_hacker_news_top_stories]
print("✅ Ohita tool defined and ready to use.")

In [ ]:
from langchain_openai import ChatOpenAI
from langchain.agents import create_tool_calling_agent, AgentExecutor
from langchain_core.prompts import ChatPromptTemplate

# 1. Initialize the LLM
llm = ChatOpenAI(model="gpt-4o", temperature=0)

# 2. Create the system prompt
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful AI assistant connected to the internet via Ohita. Use your tools to answer the user's questions."),
    ("human", "{input}"),
    ("placeholder", "{agent_scratchpad}"), # LangChain uses this to track tool outputs
])

# 3. Construct the Tool Calling Agent
agent = create_tool_calling_agent(llm, tools, prompt)

# 4. Create the Executor (verbose=True lets us watch the magic happen)
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

# 5. Run it!
print("🚀 Sending prompt to the LLM...\n")
response = agent_executor.invoke({
    "input": "What are the top 3 stories on Hacker News right now? Please summarize them briefly."
})

print("\n🎉 Final Output Generated Successfully!")